# NBA Draft Combine Data

This is a notebook that will obtain a dataset that has all of the NBA Draft Combine Data for players from 2000 to 2025

Source: [NBA Draft Combine](https://www.kaggle.com/datasets/marcusfern/nba-draft-combine)

Authors: Rasheed Martin

## Import the data using Kaggle API

Kaggle offers a very helpful api to download datasets from and made it very easy to obtain this data in a structured format.

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("marcusfern/nba-draft-combine")

print("Path to dataset files:", path)

Path to dataset files: /Users/rashe/.cache/kagglehub/datasets/marcusfern/nba-draft-combine/versions/15


## Copying files over to local repo location
 Kaggle's API downloads files into a cache folder. Using `shutil`, we can copy the files to our local repo location. 

In [2]:
import shutil 
from pathlib import Path
dir = Path("../data/raw_data/NBA_Draft_Combine/")
dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(path, dir, dirs_exist_ok=True)

PosixPath('../data/raw_data/NBA_Draft_Combine')

## Clean the Data

In the next few cells, we will clean and format the data

In [1]:
import polars as pl
from pathlib import Path
dir = Path.cwd().parent.parent

df = pl.read_csv(Path(dir / "data/raw_data/NBA_Draft_Combine/Draft Combine - Kaggle.csv", null_values="NA"))

/var/folders/k0/7nbjk7js45l8wk701f59my980000gn/T/ipykernel_20863/3476525241.py:5: DeprecationWarning: support for supplying keyword arguments to pathlib.PurePath is deprecated and scheduled for removal in Python 3.14
  df = pl.read_csv(Path(dir / "data/raw_data/NBA_Draft_Combine/Draft Combine - Kaggle.csv", null_values="NA"))


ComputeError: could not parse `NA` as dtype `f64` at column 'WGT' (column number 5)

The current offset in the file is 3155 bytes.

You might want to try:
- increasing `infer_schema_length` (e.g. `infer_schema_length=10000`),
- specifying correct dtype with the `schema_overrides` argument
- setting `ignore_errors` to `True`,
- adding `NA` to the `null_values` list.

Original error: ```remaining bytes non-empty```

In [4]:
df

YEAR,PLAYER,POS,HGT,WGT,BMI,BF,WNGSPN,STNDRCH,HANDL,HANDW,STNDVERT,LPVERT,LANE,SHUTTLE,SPRINT,BENCH,BAR,PAN,PBHGT,PDHGT
i64,str,str,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64
2025,"""Almansa, Izan""","""C""",81.25,220.4,23.5,null,85.75,109.5,9.25,9.5,28.0,34.0,11.5,2.9,3.3,null,"""105.5%""",87.9,137.5,143.5
2025,"""Avdalas, Neoklis""","""SF""",79.5,197.8,22.0,null,81.0,104.5,8.25,9.8,27.0,33.5,11.0,3.0,3.2,null,"""101.9%""",80.9,131.5,138.0
2025,"""Bailey, Ace""","""SF-PF""",79.5,203.0,22.6,null,84.5,107.0,8.5,9.0,28.0,35.0,11.0,3.0,3.0,null,"""106.3%""",76.5,135.0,142.0
2025,"""Bates, Tamar""","""SG""",75.75,191.0,23.4,null,82.25,103.0,9.75,9.0,28.0,34.0,11.0,3.0,3.0,null,"""108.6%""",87.8,131.0,137.0
2025,"""Brea, Koby""","""SG""",77.75,202.0,23.5,null,77.25,101.5,8.5,9.3,28.0,34.0,11.0,3.0,3.0,null,"""99.4%""",79.1,129.5,135.5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2000,"""Smith, Mike""","""SF""",78.5,198.0,22.6,null,80.5,103.0,null,null,33.0,21.0,11.32,null,3.22,"""12""","""102.5%""",null,136.0,124.0
2000,"""Stephens, Jarrett""","""PF""",77.25,258.0,30.4,null,81.25,103.0,null,null,27.0,29.0,11.98,null,3.46,"""13""","""105.2%""",null,130.0,132.0
2000,"""Thornton, Bootsy""","""SG""",75.5,194.5,24.0,null,78.5,101.5,null,null,26.5,31.0,12.08,null,3.33,"""7""","""104.0%""",null,128.0,132.5


In [5]:
# Convert the columns to Title Case
df.columns = list(map(str.title, df.columns))
df.columns

['Year',
 'Player',
 'Pos',
 'Hgt',
 'Wgt',
 'Bmi',
 'Bf',
 'Wngspn',
 'Stndrch',
 'Handl',
 'Handw',
 'Stndvert',
 'Lpvert',
 'Lane',
 'Shuttle',
 'Sprint',
 'Bench',
 'Bar',
 'Pan',
 'Pbhgt',
 'Pdhgt']

### Fix formatting of Player Name

The player name included things such as "*" and ".". So we will remove those elements and make sure the names have first and last

In [6]:
df = df.with_columns(
    pl.col("Player")
    .str.replace_all(".", "", literal=True)
    .str.replace_all("*", "", literal=True)
    .str.split_exact(", ", 1)
    .struct.rename_fields(["last", "first"])
    .map_elements(lambda s: f"{s['first']} {s['last']}", return_dtype=pl.String)
)

In [7]:
df.columns

['Year',
 'Player',
 'Pos',
 'Hgt',
 'Wgt',
 'Bmi',
 'Bf',
 'Wngspn',
 'Stndrch',
 'Handl',
 'Handw',
 'Stndvert',
 'Lpvert',
 'Lane',
 'Shuttle',
 'Sprint',
 'Bench',
 'Bar',
 'Pan',
 'Pbhgt',
 'Pdhgt']

Rename columns to more readable names. This will make the EDA easier in the future


In [8]:
df = df.rename({
    "Pos": "Position",
    "Hgt": "Height",
    "Wgt": "Weight",
    "Bf": "Body_Fat",
    "Wngspn": "Wingspan",
    "Stndrch": "Standing_Reach",
    "Handl": "Length_of_Hand",
    "Handw": "Width_of_Hand",
    'Stndvert': "Standing_Vertical",
     'Lpvert': "Max_Vertical_Leap",
     'Lane': "Lane_Agility_Time",
     'Shuttle': "Shuttle_Run",
     'Sprint': "3_Quarter_Sprint",
     'Bench': "Max_Bench_Press",
     'Bar': "Ration_of_Wingspan",
     'Pan': "Length_x_Width_Hand",
     'Pbhgt': "Standing_Reach_Vertical",
     'Pdhgt': "Standing_Reach_Max_Vertical"
})
df

Year,Player,Position,Height,Weight,Bmi,Body_Fat,Wingspan,Standing_Reach,Length_of_Hand,Width_of_Hand,Standing_Vertical,Max_Vertical_Leap,Lane_Agility_Time,Shuttle_Run,3_Quarter_Sprint,Max_Bench_Press,Ration_of_Wingspan,Length_x_Width_Hand,Standing_Reach_Vertical,Standing_Reach_Max_Vertical
i64,str,str,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64
2025,"""Izan Almansa""","""C""",81.25,220.4,23.5,null,85.75,109.5,9.25,9.5,28.0,34.0,11.5,2.9,3.3,null,"""105.5%""",87.9,137.5,143.5
2025,"""Neoklis Avdalas""","""SF""",79.5,197.8,22.0,null,81.0,104.5,8.25,9.8,27.0,33.5,11.0,3.0,3.2,null,"""101.9%""",80.9,131.5,138.0
2025,"""Ace Bailey""","""SF-PF""",79.5,203.0,22.6,null,84.5,107.0,8.5,9.0,28.0,35.0,11.0,3.0,3.0,null,"""106.3%""",76.5,135.0,142.0
2025,"""Tamar Bates""","""SG""",75.75,191.0,23.4,null,82.25,103.0,9.75,9.0,28.0,34.0,11.0,3.0,3.0,null,"""108.6%""",87.8,131.0,137.0
2025,"""Koby Brea""","""SG""",77.75,202.0,23.5,null,77.25,101.5,8.5,9.3,28.0,34.0,11.0,3.0,3.0,null,"""99.4%""",79.1,129.5,135.5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2000,"""Mike Smith""","""SF""",78.5,198.0,22.6,null,80.5,103.0,null,null,33.0,21.0,11.32,null,3.22,"""12""","""102.5%""",null,136.0,124.0
2000,"""Jarrett Stephens""","""PF""",77.25,258.0,30.4,null,81.25,103.0,null,null,27.0,29.0,11.98,null,3.46,"""13""","""105.2%""",null,130.0,132.0
2000,"""Bootsy Thornton""","""SG""",75.5,194.5,24.0,null,78.5,101.5,null,null,26.5,31.0,12.08,null,3.33,"""7""","""104.0%""",null,128.0,132.5


### Save Data
Save the refined data to our local storage

In [9]:
dirpath = Path("../data/refined_raw_data/NBA_Draft_Combine_Data")
dirpath.mkdir(parents=True, exist_ok=True)
filepath = Path(dirpath / "NBA_Draft_Combine_2000_2025.csv")
if dirpath.exists():
    df.write_csv(filepath, separator=",")

In [10]:
df.columns

['Year',
 'Player',
 'Position',
 'Height',
 'Weight',
 'Bmi',
 'Body_Fat',
 'Wingspan',
 'Standing_Reach',
 'Length_of_Hand',
 'Width_of_Hand',
 'Standing_Vertical',
 'Max_Vertical_Leap',
 'Lane_Agility_Time',
 'Shuttle_Run',
 '3_Quarter_Sprint',
 'Max_Bench_Press',
 'Ration_of_Wingspan',
 'Length_x_Width_Hand',
 'Standing_Reach_Vertical',
 'Standing_Reach_Max_Vertical']